# DataHelper notebook

Interactive notebook for testing the thin `DataHelper` compatibility layer built on top of `DataGateway`.

## 1. Environment setup

In [1]:
import os
import sys
import datetime as dt
from pathlib import Path
from tempfile import TemporaryDirectory

sys.path.insert(0, os.path.abspath("../src"))

import pandas as pd
from dask.distributed import LocalCluster
from sqlalchemy import Date, String, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

from boti_data import DataHelper

## 2. Define temporary SQL models

In [2]:
class Base(DeclarativeBase):
    pass


class LegacyProduct(Base):
    __tablename__ = "legacy_products"

    id: Mapped[int] = mapped_column(primary_key=True)
    id_tipo_produto: Mapped[int]
    codigo_barra: Mapped[str] = mapped_column(String(32))
    nombre_th: Mapped[str] = mapped_column(String(64))


class Event(Base):
    __tablename__ = "events"

    id: Mapped[int] = mapped_column(primary_key=True)
    event_date: Mapped[dt.date] = mapped_column(Date())
    status: Mapped[str] = mapped_column(String(32))


class User(Base):
    __tablename__ = "users"

    id: Mapped[int] = mapped_column(primary_key=True)
    status: Mapped[str] = mapped_column(String(32))


class UserProfile(Base):
    __tablename__ = "user_profiles"

    id: Mapped[int] = mapped_column(primary_key=True)
    tier: Mapped[str] = mapped_column(String(32))

## 3. Seed a temporary SQLite database

In [3]:
tmp_dir = TemporaryDirectory()
db_path = Path(tmp_dir.name) / "data_helper_demo.db"
engine = create_engine(f"sqlite:///{db_path}")

Base.metadata.create_all(engine)

with Session(engine) as session:
    session.add_all(
        [
            LegacyProduct(id_tipo_produto=1, codigo_barra="A-001", nombre_th="Alice"),
            LegacyProduct(id_tipo_produto=1, codigo_barra="A-002", nombre_th="Ana"),
            LegacyProduct(id_tipo_produto=2, codigo_barra="B-001", nombre_th="Bob"),
        ]
    )
    session.add_all(
        [
            Event(event_date=dt.date(2026, 1, 1), status="active"),
            Event(event_date=dt.date(2026, 1, 2), status="inactive"),
        ]
    )
    session.add_all(
        [
            User(id=1, status="active"),
            User(id=2, status="inactive"),
            User(id=3, status="active"),
            User(id=4, status="active"),
        ]
    )
    session.add_all(
        [
            UserProfile(id=1, tier="gold"),
            UserProfile(id=3, tier="standard"),
            UserProfile(id=4, tier="gold"),
        ]
    )
    session.commit()

db_url = f"sqlite:///{db_path}"
db_url

'sqlite:////var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/tmpuqt00nff/data_helper_demo.db'

## 4. Test legacy-style configured SQL usage

In [4]:
legacy_helper = DataHelper.from_legacy_config(
    {
        "backend": "sqlalchemy",
        "connection_url": db_url,
        "poolclass": "sqlalchemy.pool.NullPool",
        "query_only": False,
        "table": "legacy_products",
        "field_map": {
            "id_tipo_produto": "product_type_id",
            "codigo_barra": "barcode",
            "nombre_th": "first_name",
        },
        "sticky_filters": {"product_type_id": 1},
        "df_params": {"fieldnames": ("product_type_id", "barcode", "first_name")},
        "df_options": {"sort_field": "barcode"},
    }
)

legacy_frame = legacy_helper.load(as_pandas=True)
legacy_frame

,product_type_id,barcode,first_name
0,1,A-001,Alice
1,1,A-002,Ana


## 5. Test period loading

In [5]:
period_helper = DataHelper(
    {
        "backend": "sqlalchemy",
        "connection_url": db_url,
        "poolclass": "sqlalchemy.pool.NullPool",
        "query_only": False,
        "table": "events",
        "df_params": {"fieldnames": ("event_date", "status")},
    }
)

period_frame = period_helper.load_period("event_date", "2026-01-01", "2026-01-01", as_pandas=True)
period_frame

,event_date,status
0,2026-01-01 00:00:00+00:00,active


## 6. Test distributed session and helper join convenience

In [6]:
distributed_helper = DataHelper(
    {
        "backend": "sqlalchemy",
        "connection_url": db_url,
        "poolclass": "sqlalchemy.pool.NullPool",
        "query_only": False
    }
)

with LocalCluster(n_workers=2, threads_per_worker=1, processes=False, dashboard_address=None) as cluster:
    with DataHelper.session(scheduler_address=cluster.scheduler_address):
        users = distributed_helper.load(statement=select(User), model=User)
        profiles = distributed_helper.load(statement=select(UserProfile), model=UserProfile)
        joined = DataHelper.left_join(
            users,
            profiles,
            join_key="id",
            join_schema_map={"id": "Int64"},
            persist=True,
        )
        joined_frame = joined.compute().sort_values("id").reset_index(drop=True)

joined_frame

/Users/lvalverdeb/TeamDev/repo-split/boti-data/src/boti_data/db/partitioned_loader.py:36: UserWarning: Distributed SQL task payloads will carry the raw DSN credential because 'worker_connection_env_var' is not set on SqlDatabaseConfig. Set 'worker_connection_env_var' to the name of an environment variable that resolves the DSN on each worker to avoid serializing credentials.
  self._worker_config = WorkerSqlConfig.from_database_config(config)
WorkerSqlConfig created with raw DSN fallback; set worker_connection_env_var to avoid credential serialization.
/Users/lvalverdeb/TeamDev/repo-split/boti-data/src/boti_data/db/partitioned_loader.py:36: UserWarning: Distributed SQL task payloads will carry the raw DSN credential because 'worker_connection_env_var' is not set on SqlDatabaseConfig. Set 'worker_connection_env_var' to the name of an environment variable that resolves the DSN on each worker to avoid serializing credentials.
  self._worker_config = WorkerSqlConfig.from_database_config(co

,id,status,tier
0,1,active,gold
1,2,inactive,<NA>
2,3,active,standard
3,4,active,gold


## 7. Regression — filter/coercion bug fixes

Tests for five silent-correctness bugs found in the boti-data package:

| # | Bug | Where |
|---|---|---|
| A | `align_in_types` silently converted numeric column to strings on coercion failure | `filters/utils.py` |
| B | `coerce_arrow_table` used `safe=False` fallback, silently overflowing integers | `db/arrow_schema_mapper.py` |
| C | `as_str()` filled NULL with `""`, causing NULL rows to match empty-string filters | `filters/utils.py` |
| D | `_coerce_value` truncated fractional floats to int without error (`3.99 → 3`) | `db/arrow_schema_mapper.py` |
| E | Bad date in `date__gt` filter was silently swallowed instead of raising immediately | `filters/handler.py` |

In [7]:
import dask.dataframe as dd
import pyarrow as pa

from boti_data.db.arrow_schema_mapper import _coerce_value, coerce_arrow_table
from boti_data.filters.handler import FilterHandler
from boti_data.filters.utils import align_in_types, as_str

all_ok = True

# ---------------------------------------------------------------------------
# Bug A — align_in_types raises TypeError instead of silently falling to strings
# ---------------------------------------------------------------------------
# Old: int column + ["abc"] → str column + ["abc"] → isin always False, no error.
# New: raises TypeError with a clear message.
try:
    int_col = dd.from_pandas(pd.DataFrame({'id': [1, 2, 3]}), npartitions=1)['id']
    align_in_types(int_col, ['abc', 'def'])
    print('✗ Bug A FAIL: should have raised TypeError')
    all_ok = False
except TypeError:
    print('✓ Bug A: align_in_types raises TypeError for non-coercible IN values')

# Sanity-check: valid int values still work
aligned_col, aligned_vals = align_in_types(int_col, [1, 3])
assert aligned_vals == [1, 3], f'Expected [1, 3], got {aligned_vals}'
print('  (sanity) valid int IN filter still works')

# ---------------------------------------------------------------------------
# Bug B — coerce_arrow_table raises instead of silent safe=False overflow
# ---------------------------------------------------------------------------
# Old: "99999999999999999999" cast to int64 with safe=False → undefined overflow.
# New: raises ArrowInvalid with a clear message.
try:
    t_overflow = pa.table({'val': pa.array(['99999999999999999999'])})
    coerce_arrow_table(t_overflow, pa.schema([pa.field('val', pa.int64())]))
    print('✗ Bug B FAIL: should have raised ArrowInvalid')
    all_ok = False
except pa.lib.ArrowInvalid:
    print('✓ Bug B: coerce_arrow_table raises ArrowInvalid instead of silent overflow')

# Sanity-check: valid string → int64 cast still works
t_ok = pa.table({'val': pa.array(['42'])})
result = coerce_arrow_table(t_ok, pa.schema([pa.field('val', pa.int64())]))
assert result.column('val').to_pylist() == [42]
print('  (sanity) valid string→int64 cast still works')

# ---------------------------------------------------------------------------
# Bug C — as_str() no longer fills NULL with "" (NULL should not match "" in iexact)
# ---------------------------------------------------------------------------
# Old: as_str fills NULL → ""; `iexact("")` matched NULL rows.
# New: NULL stays NA; `iexact("")` never matches NULL rows.
df_nulls = dd.from_pandas(
    pd.DataFrame({'name': pd.array(['alice', None, 'bob'], dtype='string')}),
    npartitions=1,
)
str_col = as_str(df_nulls['name'])
iexact_result = (str_col.str.lower() == '').compute()
null_row_val = iexact_result.iloc[1]  # the NULL row
null_matches_empty = bool(null_row_val) if not pd.isna(null_row_val) else False
if null_matches_empty:
    print('✗ Bug C FAIL: NULL matched empty-string iexact')
    all_ok = False
else:
    print('✓ Bug C: NULL in string column does not match iexact("") after as_str fix')

# ---------------------------------------------------------------------------
# Bug D — _coerce_value raises for fractional floats targeting integer types
# ---------------------------------------------------------------------------
# Old: _coerce_value(3.99, int64) → int(3.99) = 3, silently loses .99.
# New: raises ValueError.
try:
    _coerce_value(3.99, pa.int64())
    print('✗ Bug D FAIL: should have raised ValueError')
    all_ok = False
except ValueError:
    print('✓ Bug D: _coerce_value raises ValueError for fractional float → int64')

# Lossless integer float (3.0 → 3) must still work
assert _coerce_value(3.0, pa.int64()) == 3
print('  (sanity) _coerce_value(3.0, int64) = 3 (lossless conversion still works)')

# ---------------------------------------------------------------------------
# Bug E — bad date in date__gt filter raises immediately
# ---------------------------------------------------------------------------
# Old: pd.Timestamp("not-a-date") raised, was caught, filter silently moved to
#      residual — where it would raise later with a confusing traceback.
# New: ValueError propagates immediately from split_pushdown_and_residual.
fh = FilterHandler('dask')
try:
    fh.split_pushdown_and_residual({'created_at__date__gt': 'not-a-date'})
    print('✗ Bug E FAIL: should have raised ValueError')
    all_ok = False
except ValueError:
    print('✓ Bug E: invalid date filter raises ValueError immediately from FilterHandler')

print()
print('✓ All regression tests passed' if all_ok else '✗ Some regression tests FAILED')

✓ Bug A: align_in_types raises TypeError for non-coercible IN values
  (sanity) valid int IN filter still works
✓ Bug B: coerce_arrow_table raises ArrowInvalid instead of silent overflow
  (sanity) valid string→int64 cast still works
✓ Bug C: NULL in string column does not match iexact("") after as_str fix
✓ Bug D: _coerce_value raises ValueError for fractional float → int64
  (sanity) _coerce_value(3.0, int64) = 3 (lossless conversion still works)
✓ Bug E: invalid date filter raises ValueError immediately from FilterHandler

✓ All regression tests passed


## 7. Cleanup

In [8]:
legacy_helper.close()
period_helper.close()
distributed_helper.close()
engine.dispose()
tmp_dir.cleanup()